<a href="https://colab.research.google.com/github/CraigUni27/Python-Practice/blob/main/OEE_All_Tag_Charts_v1_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OEE / HydraRacker — Full-width stacked tag charts

Use this notebook with:

- `tags_2026-09-30_HR1_st`

It creates **one large chart per tag, stacked vertically**, with titles formatted as **TAG — Description**.  
The output is interactive: zoom, pan, hover values, and jump-to-tag links.

Run the notebook in Google Colab using **Runtime → Run all**, then upload the two source files when prompted.

In [13]:
!pip -q install plotly openpyxl xlsxwriter

import io
import re
import html
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from google.colab import files
from IPython.display import display

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.5 MB/s eta 0:00:00


## 1. Upload the CSV and tag-list workbook

In [24]:
from google.colab import files
import pandas as pd

# 1. Prompt to select and upload the file from your local PC
uploaded = files.upload()

# 2. Get the filename of the uploaded file
filename = list(uploaded.keys())[0]

# 3. Read the Excel file into a DataFrame
df = pd.read_excel(filename)

# Display the first few rows
df.head()

Saving tags_2026-09-30_HR1_st.xlsx to tags_2026-09-30_HR1_st (1).xlsx


,topic,description,Short Description,time,value,Converted Time
0,forth-mqtt/ns2_Objects_NOV_Signals_HR1-203-A00-st,Actual Position,Main Arm Actual Position,1790779979972,0.995279,2026-09-30 14:52:59.972
1,forth-mqtt/ns2_Objects_NOV_Signals_HR1-181-A00-st,Actual Load,Actual Load,1790779979972,-0.219553,2026-09-30 14:52:59.972
2,forth-mqtt/ns2_Objects_NOV_Signals_HR1-352-A00-st,Actual Position,Lower Trolley Actual Position,1790779979675,4.015215,2026-09-30 14:52:59.675
3,forth-mqtt/ns2_Objects_NOV_Signals_HR1-333-A00-st,Actual Position,Upper Trolley Actual Position,1790779979675,4.015093,2026-09-30 14:52:59.675
4,forth-mqtt/ns2_Objects_NOV_Signals_HR1-522-A00-st,Actual Grip Pressure,Main Arm Roller Close Pressure Actual Value,1790779979221,45.601852,2026-09-30 14:52:59.221


## 2. Read and clean the CSV

In [32]:
raw = pd.read_excel(filename)

required = {"topic", "time", "value"}
missing = required - set(raw.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {sorted(missing)}")

def extract_tag(topic):
    s = str(topic)
    if "_Signals_" in s:
        return s.split("_Signals_", 1)[1]
    return re.split(r"[/_]", s)[-1]

raw["Tag"] = raw["topic"].map(extract_tag).astype(str).str.strip()
raw["Time BST"] = (
    pd.to_datetime(raw["time"], unit="ms", utc=True, errors="coerce")
      .dt.tz_convert("Europe/London")
)

def value_to_number(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s.lower() == "true":
        return 1.0
    if s.lower() == "false":
        return 0.0
    try:
        return float(s)
    except Exception:
        return np.nan

raw["Value_numeric"] = raw["value"].map(value_to_number)

data = (
    raw.loc[raw["Time BST"].notna(),
            ["Tag", "description", "Time BST", "value", "Value_numeric"]]
       .sort_values(["Tag", "Time BST"])
       .reset_index(drop=True)
)

data["Canonical Description"] = data["Tag"].map(canonical_description)

print(f"Rows loaded: {len(data):,}")
print(f"Unique tags: {data['Tag'].nunique():,}")
print(f"Time range: {data['Time BST'].min()}  →  {data['Time BST'].max()}")

Rows loaded: 18,308
Unique tags: 68
Time range: 2026-09-30 15:00:00.060000+01:00  →  2026-09-30 15:52:59.972000+01:00


Here is a line-by-line breakdown of what this code does, broken down into five distinct steps.Step 1: Loading Data & Validating ColumnsPythonraw = pd.read_excel(filename)

required = {"topic", "time", "value"}
missing = required - set(raw.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {sorted(missing)}")
raw = pd.read_excel(filename): Reads the Excel file into a pandas DataFrame named raw.required = {"topic", "time", "value"}: Sets the minimum required column names.missing = required - set(raw.columns): Uses set subtraction to find which required columns are not present in raw.columns.if missing: raise ValueError(...): Stops the script with an error message listing the missing columns if any required columns are absent.Step 2: Extracting a Clean "Tag" NamePythondef extract_tag(topic):
    s = str(topic)
    if "_Signals_" in s:
        return s.split("_Signals_", 1)[1]
    return re.split(r"[/_]", s)[-1]

raw["Tag"] = raw["topic"].map(extract_tag).astype(str).str.strip()
extract_tag(topic): Extracts a simplified tag name from a long topic string:If "_Signals_" is in the topic string, it takes everything after "_Signals_".Otherwise, it splits the string by slashes (/) or underscores (_) and grabs the very last piece ([-1]).raw["Tag"] = ...: Applies extract_tag across every row in the topic column, converts the result to a string, and strips trailing/leading whitespace.Step 3: Converting Epoch Timestamps to UK TimePythonraw["Time BST"] = (
    pd.to_datetime(raw["time"], unit="ms", utc=True, errors="coerce")
      .dt.tz_convert("Europe/London")
)
unit="ms", utc=True: Converts numerical millisecond Unix timestamps in the time column into UTC datetime objects.errors="coerce": Converts invalid/corrupted timestamps into NaT (Not a Time/blank) instead of throwing an error..dt.tz_convert("Europe/London"): Converts UTC time to UK local time (Europe/London), automatically adjusting for daylight saving time (GMT/BST).Step 4: Cleaning & Converting Values to NumbersPythondef value_to_number(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s.lower() == "true":
        return 1.0
    if s.lower() == "false":
        return 0.0
    try:
        return float(s)
    except Exception:
        return np.nan

raw["Value_numeric"] = raw["value"].map(value_to_number)
value_to_number(v): Converts mixed data types in the value column into clean numbers:Missing values (NaN) return np.nan.Text booleans "true"/"false" (case-insensitive) map to 1.0 and 0.0.Numbers or numeric strings convert to floats (e.g., "42.5" $\rightarrow$ 42.5).Unconvertible text (like "Error") returns np.nan.raw["Value_numeric"] = ...: Stores these standardized numbers in a new column.Step 5: Filtering, Sorting, & Summary DisplayPythondata = (
    raw.loc[raw["Time BST"].notna(),
            ["Tag", "description", "Time BST", "value", "Value_numeric"]]
       .sort_values(["Tag", "Time BST"])
       .reset_index(drop=True)
)

print(f"Rows loaded: {len(data):,}")
print(f"Unique tags: {data['Tag'].nunique():,}")
print(f"Time range: {data['Time BST'].min()}  →  {data['Time BST'].max()}")
raw.loc[...]: Filters out any rows where Time BST failed to convert, keeping only specified columns (Tag, description, Time BST, value, Value_numeric)..sort_values(...): Sorts the dataset alphabetically by Tag, then chronologically by Time BST..reset_index(drop=True): Re-indexes the rows starting from 0.print(...): Prints formatted summary metrics (total rows loaded, total distinct tags, and earliest to latest timestamp).

## 4. Chart settings

The default chart size is deliberately large and readable, similar to the `New One Sequence` chart size.

In [26]:
CHART_HEIGHT = 360
CHART_WIDTH = 1180

# To restrict the report to HR1 only, replace the next line with:
# TAG_FILTER = lambda t: t.startswith("HR1-")
TAG_FILTER = lambda t: True

tags = sorted(t for t in data["Tag"].dropna().unique() if TAG_FILTER(t))
print(f"Charts to create: {len(tags)}")

Charts to create: 68


## 5. Create the stacked charts

In [27]:
figures = []
summary_rows = []

def is_digital(series):
    vals = pd.to_numeric(series, errors="coerce").dropna()
    if vals.empty:
        return False
    uniq = set(np.round(vals.unique(), 10).tolist())
    return len(uniq) <= 2 and uniq.issubset({0.0, 1.0})

for tag in tags:
    d = data.loc[data["Tag"] == tag].copy()
    desc = canonical_description(tag)

    digital = is_digital(d["Value_numeric"])

    if d["Value_numeric"].notna().sum() == 0:
        states = list(pd.Series(d["value"].astype(str)).drop_duplicates())
        state_map = {state: i for i, state in enumerate(states)}
        d["PlotValue"] = d["value"].astype(str).map(state_map)
        y_title = "State"
        customdata = d["value"].astype(str)
        hover_value = "<br>State: %{customdata}"
        line_shape = "hv"
    else:
        d["PlotValue"] = d["Value_numeric"]
        y_title = "Value"
        customdata = None
        hover_value = "<br>Value: %{y:.6g}"
        line_shape = "hv" if digital else "linear"

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=d["Time BST"],
            y=d["PlotValue"],
            mode="lines",
            line_shape=line_shape,
            name=tag,
            customdata=customdata,
            hovertemplate=(
                "<b>" + html.escape(tag) + "</b>"
                + "<br>%{x|%H:%M:%S.%L}"
                + hover_value
                + "<extra></extra>"
            ),
        )
    )

    fig.update_layout(
        title=dict(
            text=f"<b>{html.escape(tag)}</b> — {html.escape(str(desc))}",
            x=0.01,
            xanchor="left",
        ),
        height=CHART_HEIGHT,
        width=CHART_WIDTH,
        margin=dict(l=75, r=35, t=65, b=55),
        showlegend=False,
        hovermode="x",
        template="plotly_white",
    )
    fig.update_xaxes(title_text="Time BST", showgrid=True)
    fig.update_yaxes(title_text=y_title, showgrid=True)

    if digital:
        fig.update_yaxes(
            tickmode="array",
            tickvals=[0, 1],
            ticktext=["False / 0", "True / 1"],
            range=[-0.15, 1.15],
        )

    figures.append((tag, desc, fig))

    valid = pd.to_numeric(d["PlotValue"], errors="coerce").dropna()
    summary_rows.append({
        "Tag": tag,
        "Description": desc,
        "Samples": len(d),
        "First time BST": d["Time BST"].min(),
        "Last time BST": d["Time BST"].max(),
        "Min": valid.min() if len(valid) else np.nan,
        "Max": valid.max() if len(valid) else np.nan,
        "Digital/Boolean": digital,
    })

for _, _, fig in figures:
    fig.show()

summary = pd.DataFrame(summary_rows)
display(summary)

,Tag,Description,Samples,First time BST,Last time BST,Min,Max,Digital/Boolean
0,HR1-001-D00-st,Close Main Arm Roller,40,2026-09-30 15:00:56.139000+01:00,2026-09-30 15:52:15.346000+01:00,0.0,1.0,True
1,HR1-002-D00-st,Open Main Arm Roller,40,2026-09-30 15:00:54.717000+01:00,2026-09-30 15:52:15.862000+01:00,0.0,1.0,True
2,HR1-004-D00-st,Close Main Arm Gripper,40,2026-09-30 15:00:57.623000+01:00,2026-09-30 15:52:14.862000+01:00,0.0,1.0,True
3,HR1-005-D00-st,Open Main Arm Gripper,40,2026-09-30 15:00:56.139000+01:00,2026-09-30 15:52:15.346000+01:00,0.0,1.0,True
4,HR1-010-D00-st,Close Tail Arm Guide,40,2026-09-30 15:00:55.248000+01:00,2026-09-30 15:52:15.346000+01:00,0.0,1.0,True
...,...,...,...,...,...,...,...,...
63,HR1-635-D01-st,"Main well Tubular Chute, Upper Guide Close",40,2026-09-30 15:00:27.201000+01:00,2026-09-30 15:51:02.719000+01:00,0.0,1.0,True
64,HR1-635-D04-st,"Main well Tubular Chute, Rabbit in Target",40,2026-09-30 15:00:13.701000+01:00,2026-09-30 15:51:30.219000+01:00,0.0,1.0,True
65,HR1-638-D02-st,Common Status Alarm Upper Arm,16,2026-09-30 15:00:01.201000+01:00,2026-09-30 15:35:50.205000+01:00,0.0,1.0,True
66,HR1-638-D03-st,Common Status Alarm Main Arm,34,2026-09-30 15:00:01.201000+01:00,2026-09-30 15:52:21.206000+01:00,0.0,1.0,True


## 6. Export one standalone interactive HTML report

This is the recommended review format: one large chart per tag, vertical scrolling, zoom/pan, hover values, and a clickable tag index.

In [33]:
report_name = "HydraRacker_All_Tag_Charts_Stacked.html"

index_links = []
chart_blocks = []

for i, (tag, desc, fig) in enumerate(figures):
    anchor = re.sub(r"[^A-Za-z0-9_-]+", "-", tag)

    index_links.append(
        '<a href="#{}"><b>{}</b> — {}</a>'.format(
            anchor,
            html.escape(tag),
            html.escape(str(desc)),
        )
    )

    block = pio.to_html(
        fig,
        full_html=False,
        include_plotlyjs="cdn" if i == 0 else False,
        config={
            "displaylogo": False,
            "scrollZoom": True,
            "responsive": True,
        },
    )
    chart_blocks.append(
        '<section class="chart-section" id="{}">{}</section>'.format(anchor, block)
    )

page = '''<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>HydraRacker - All Tag Charts</title>
<style>
body { font-family: Arial, Helvetica, sans-serif; margin: 0; background: #f4f6f8; color: #18212b; }
.page { max-width: 1240px; margin: 0 auto; padding: 24px 26px 60px 26px; }
h1 { margin-bottom: 6px; }
.subtitle { color: #53606d; margin-bottom: 20px; }
.index { background: white; border: 1px solid #d9e0e6; border-radius: 8px; padding: 18px 22px; margin-bottom: 24px; columns: 2; column-gap: 30px; }
.index a { display: block; color: #183b56; text-decoration: none; padding: 4px 0; break-inside: avoid; }
.index a:hover { text-decoration: underline; }
.chart-section { background: white; border: 1px solid #d9e0e6; border-radius: 8px; margin: 18px 0; padding: 8px; overflow-x: auto; }
.back { position: fixed; right: 18px; bottom: 18px; background: #183b56; color: white; border-radius: 20px; padding: 9px 14px; text-decoration: none; font-size: 13px; }
</style>
</head>
<body id="top">
<div class="page">
    <h1>HydraRacker - All Tag Charts</h1>
    <div class="subtitle">Full-width stacked plots. Time shown in Europe/London.</div>
    <div class="index">__INDEX__</div>
    __CHARTS__
</div>
<a class="back" href="#top">Back to top</a>
</body>
</html>'''

page = page.replace("__INDEX__", "".join(index_links))
page = page.replace("__CHARTS__", "".join(chart_blocks))

Path(report_name).write_text(page, encoding="utf-8")
print(f"Created: {report_name}")

Created: HydraRacker_All_Tag_Charts_Stacked.html


## 7. Save a compact tag summary

In [34]:
summary_name = "HydraRacker_All_Tag_Chart_Summary.xlsx"

# Excel does not support datetimes with timezones, so remove the timezone.
summary_to_save = summary.copy()
summary_to_save['First time BST'] = summary_to_save['First time BST'].dt.tz_localize(None)
summary_to_save['Last time BST'] = summary_to_save['Last time BST'].dt.tz_localize(None)

summary_to_save.to_excel(summary_name, index=False)
print(f"Created: {summary_name}")

Created: HydraRacker_All_Tag_Chart_Summary.xlsx


## 8. Optional: create the simplified Excel version

This creates the Excel layout you originally asked for:

- **Charts** — one large chart per tag, stacked vertically.
- **Data** — the source chart data only; hidden to keep the workbook uncluttered.
- No other analysis tabs are carried across.

The chart title is **TAG — Description**, and each chart is approximately the same readable scale as the `New One Sequence` view.

In [35]:
excel_name = "HydraRacker_All_Tag_Charts_Stacked.xlsx"

# Excel cannot store timezone-aware datetimes, so make a display copy without the timezone.
excel_data = data[["Time BST", "Tag", "Canonical Description", "Value_numeric", "value"]].copy()
excel_data["Time BST"] = excel_data["Time BST"].dt.tz_localize(None)
excel_data = excel_data.sort_values(["Tag", "Time BST"]).reset_index(drop=True)

with pd.ExcelWriter(
    excel_name,
    engine="xlsxwriter",
    datetime_format="hh:mm:ss.000",
) as writer:
    excel_data.to_excel(writer, sheet_name="Data", index=False)

    workbook = writer.book
    data_ws = writer.sheets["Data"]
    charts_ws = workbook.add_worksheet("Charts")

    # Put Charts first.
    charts_ws.activate()

    title_fmt = workbook.add_format({
        "bold": True,
        "font_size": 16,
        "font_color": "#17324D",
    })
    note_fmt = workbook.add_format({
        "font_size": 10,
        "font_color": "#5B6670",
    })
    header_fmt = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#17324D",
        "border": 0,
    })

    charts_ws.write("A1", "HydraRacker — All Tag Charts", title_fmt)
    charts_ws.write(
        "A2",
        "One full-width chart per tag; stacked vertically. Time shown in BST/Europe-London.",
        note_fmt,
    )

    # Small tag index at the top.
    charts_ws.write("A4", "Tag", header_fmt)
    charts_ws.write("B4", "Description", header_fmt)

    tag_ranges = {}
    for tag in tags:
        idx = excel_data.index[excel_data["Tag"] == tag].tolist()
        if idx:
            # +2 because XlsxWriter rows are zero-based and row 0 contains headers.
            start_excel_row = min(idx) + 2
            end_excel_row = max(idx) + 2
            tag_ranges[tag] = (start_excel_row, end_excel_row)

    for i, tag in enumerate(tags, start=5):
        desc = canonical_description(tag)
        charts_ws.write(i - 1, 0, tag)
        charts_ws.write(i - 1, 1, desc)

    charts_ws.set_column("A:A", 22)
    charts_ws.set_column("B:B", 48)

    # Begin charts below the index.
    start_row = max(8, 6 + len(tags))
    chart_block_rows = 24

    for chart_i, tag in enumerate(tags):
        if tag not in tag_ranges:
            continue

        desc = canonical_description(tag)
        start_excel_row, end_excel_row = tag_ranges[tag]

        chart = workbook.add_chart({"type": "line"})
        chart.add_series({
            "name": f"{tag} — {desc}",
            "categories": f"=Data!$A${start_excel_row}:$A${end_excel_row}",
            "values": f"=Data!$D${start_excel_row}:$D${end_excel_row}",
            "line": {"width": 1.5},
        })

        chart.set_title({
            "name": f"{tag} — {desc}",
            "name_font": {"size": 12, "bold": True},
        })
        chart.set_x_axis({
            "name": "Time BST",
            "num_format": "hh:mm:ss",
            "major_gridlines": {"visible": False},
        })
        chart.set_y_axis({
            "name": "Value",
            "major_gridlines": {"visible": True},
        })
        chart.set_legend({"none": True})
        chart.set_size({"width": 1180, "height": 360})
        chart.set_plotarea({
            "border": {"none": True},
        })
        chart.set_chartarea({
            "border": {"color": "#C8D1DA"},
        })

        # Digital signals get a fixed 0/1-style view.
        d = excel_data.loc[excel_data["Tag"] == tag, "Value_numeric"].dropna()
        uniq = set(np.round(d.unique(), 10).tolist()) if len(d) else set()
        if len(uniq) <= 2 and uniq.issubset({0.0, 1.0}):
            chart.set_y_axis({
                "name": "State",
                "min": -0.15,
                "max": 1.15,
                "major_unit": 1,
                "major_gridlines": {"visible": True},
            })

        anchor_row = start_row + chart_i * chart_block_rows
        charts_ws.insert_chart(anchor_row, 0, chart)

    # Keep only the chart sheet visible for normal use.
    data_ws.hide()

print(f"Created: {excel_name}")

Created: HydraRacker_All_Tag_Charts_Stacked.xlsx


## 9. Download the outputs

In [36]:
files.download(report_name)
files.download(excel_name)
files.download(summary_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>